In [ ]:
# Fix protobuf version mismatch for kaggle_benchmarks
!pip install protobuf==5.29.6 --quiet


# Trinity Social Cognition Probe — Pragmatic Inference
**Task 22 of 25** | Track 5: Social | Brain Zone: WERNICKEMeasures sarcasm/irony/implicature understanding.## Trinity Neuroanatomical Foundation
### Ternary Scoring {-1, 0, +1}- **+1**: Correct
- **0**: Partial with uncertainty
- **-1**: Wrong
### φ-Scaling
Context cues: 1, 2, 3, 4, 5

In [ ]:
import kaggle_benchmarks as kbench
import pandas as pd
df = pd.read_csv('../../data/tscp_social.csv')
prag_df = df[df['task'] == 'Pragmatic Inference'].copy()

In [ ]:
@dataclass
class PragmaticResponse:
    interpretation: str
    confidence: float
    def ternary_score(self, expected) -> int:
        if self.interpretation == expected:
            return 1
        elif 0.3 < self.confidence < 0.7:
            return 0
        else:
            return -1


In [ ]:
@kbench.task(name="trinity_wernicke_pragmatic")
def pragmatic_inference(llm, utterance, context, expected) -> float:
    prompt = f""Utterance: '{utterance}'\nContext: {context}\nQuestion: What does this mean?\nConfidence (0-1)"
    response = llm.prompt(prompt, schema=PragmaticResponse)
    ternary = response.ternary_score(expected)
    confidence_score = (response.confidence - 0.5) * 2
    final_score = 0.7 * ternary + 0.3 * confidence_score
    return max(-1.0, min(1.0, final_score))
    print("Task registered")

In [ ]:
results = pragmatic_inference.evaluate(llm=[kbench.llm], evaluation_data=prag_df.head(10))
print(results.head())

In [ ]:
# Full eval
# results = pragmatic_inference.evaluate(llm=[kbench.llm], evaluation_data=prag_df)
# kbench.submit(task=pragmatic_inference, results=results, message="Trinity Wernicke Pragmatic v1.0")
# print("Submitted!")

## Expected Leaderboard Performance
Models are expected to show clear separation on this benchmark:
| Model | Expected Score | Reason |
|-------|---------------|--------|
| GPT-4o | 0.70-0.80 | Strong pragmatic |
| Claude Sonnet | 0.65-0.75 | Good pragmatic |
| Gemini Pro | 0.60-0.70 | Moderate pragmatic |
| Llama 3 70B | 0.40-0.50 | Weak pragmatic |
The gradient demonstrates discriminatory power.